# 4.1 Sentiment Trajectory Analysis

This notebook analyzes sentiment trajectories of YouTube comments across pre-cancel, cancel, and post-apology windows.

## Research Question
*How do audience sentiment trajectories on YouTube change across pre-cancel, cancel, and post-apology windows for South Korean influencer apology videos?*

## Model Configuration
- **Model**: `monologg/koelectra-base-v3-sentiment`
- **Tokenizer**: AutoTokenizer (HuggingFace)
- **Output**: POSITIVE / NEGATIVE / NEUTRAL with confidence scores

## Phase 1: Configuration

### Model Choice Rationale
- Korean ELECTRA model fine-tuned for sentiment analysis
- Provides consistent scoring across all comments
- Handles Korean text including mixed language (Korean + English)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

def find_project_root():
    current = os.getcwd()
    while current != os.path.dirname(current):
        if os.path.exists(os.path.join(current, '02-preprocessing')):
            return current
        current = os.path.dirname(current)
    return os.getcwd()

PROJECT_DIR = find_project_root()
OUTPUT_DIR = os.path.join(PROJECT_DIR, '03-analysis', 'outputs')
FIGURE_DIR = os.path.join(PROJECT_DIR, '03-analysis', 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

PERSON = 'mimiminu'  # or 'bodeoum_tv'
MODEL_NAME = 'Copycats/koelectra-base-v3-generalized-sentiment-analysis'
BATCH_SIZE = 32

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = 'cpu' 
print(f"Using device: {device}")
print(f"Model: {MODEL_NAME}")
print(f"Person: {PERSON}")
print(f"Project root: {PROJECT_DIR}")

Using device: cuda
Model: Copycats/koelectra-base-v3-generalized-sentiment-analysis
Person: mimiminu
Project root: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis


## Phase 2: Data Loading

### Input Data
- Source: `02-preprocessing/outputs/{PERSON}/comments.csv`
- Key columns: `text_clean`, `window`, `slot`, `author`, `like_count`

### Preprocessing Steps
1. Load comments.csv
2. Use `text_clean` column (already preprocessed)
3. Remove empty/null texts
4. Deduplicate (same text in same slot = 1 comment)

In [2]:
# ============================================================
# Phase 2: Data Loading (Auto-detect available data)
# ============================================================

person_output_dir = os.path.join(OUTPUT_DIR, PERSON)

# Auto-detect latest checkpoint file
checkpoint_files = []
if os.path.exists(person_output_dir):
    checkpoint_files = [f for f in os.listdir(person_output_dir) if f.startswith('sentiment_checkpoint_') and f.endswith('.csv')]

if checkpoint_files:
    # Extract numbers and find latest
    checkpoint_nums = [int(f.replace('sentiment_checkpoint_', '').replace('.csv', '')) for f in checkpoint_files]
    latest_checkpoint = max(checkpoint_nums)
    checkpoint_path = os.path.join(person_output_dir, f'sentiment_checkpoint_{latest_checkpoint}.csv')
    print(f"Loading from checkpoint: {checkpoint_path}")
    comments_df = pd.read_csv(checkpoint_path)
    print(f"Loaded {len(comments_df)} comments from checkpoint (subset)")
else:
    # Load from original comments.csv
    comments_path = os.path.join(PROJECT_DIR, '02-preprocessing', 'outputs', PERSON, 'comments.csv')
    comments_df = pd.read_csv(comments_path)
    print(f"Loaded {len(comments_df)} comments from original file")

# Auto-detect slot and window structure from data
available_slots = sorted(comments_df['slot'].unique(), key=lambda x: (x.split('-')[0], int(x.split('-')[1]) if '-' in x else 0))
available_windows = sorted(comments_df['window'].unique(), key=lambda x: x.split('-')[0])

print(f"\nTotal comments: {len(comments_df)}")
print(f"Detected slots ({len(available_slots)}): {available_slots}")
print(f"Detected windows ({len(available_windows)}): {available_windows}")
print(f"Has sentiment data: {'sentiment_label' in comments_df.columns and 'sentiment_score' in comments_df.columns}")

print(f"\nComments per window:")
print(comments_df.groupby('window').size())

print(f"\nComments per slot:")
slot_counts = comments_df.groupby(['window', 'slot']).size().reset_index(name='count')
slot_counts = slot_counts.sort_values(['window', 'slot'])
print(slot_counts.to_string(index=False))

Loaded 16748 comments from original file

Total comments: 16748
Detected slots (21): ['cancel-1', 'post-1', 'post-2', 'post-3', 'post-4', 'post-5', 'post-6', 'post-7', 'post-8', 'post-9', 'post-10', 'pre-1', 'pre-2', 'pre-3', 'pre-4', 'pre-5', 'pre-6', 'pre-7', 'pre-8', 'pre-9', 'pre-10']
Detected windows (3): ['cancel', 'post', 'pre']
Has sentiment data: False

Comments per window:
window
cancel    3013
post      5565
pre       8170
dtype: int64

Comments per slot:
window     slot  count
cancel cancel-1   3013
  post   post-1    983
  post  post-10    716
  post   post-2    385
  post   post-3    513
  post   post-4    141
  post   post-5    466
  post   post-6    658
  post   post-7    260
  post   post-8    981
  post   post-9    462
   pre    pre-1    295
   pre   pre-10    649
   pre    pre-2    430
   pre    pre-3   1350
   pre    pre-4    706
   pre    pre-5   1545
   pre    pre-6   1026
   pre    pre-7   1035
   pre    pre-8    448
   pre    pre-9    686


## Phase 3: Model Loading & Sentiment Scoring

### Model Loading
- Load tokenizer and model from HuggingFace
- Use GPU if available (cuda)

### Scoring Strategy
- Batch processing for efficiency
- Process comments in chunks
- Store: sentiment_label, sentiment_score

In [3]:
# ============================================================
# Phase 3: Model Loading & Sentiment Scoring
# ============================================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print(f"\nModel label mapping: {model.config.id2label}")
print(f"Number of labels: {model.config.num_labels}")

CONFIDENCE_THRESHOLD = 0.6
NEUTRAL_LOW = 0.4

def convert_to_3class(pred_idx, confidence):
    if model.config.num_labels == 2:
        if pred_idx == 0:
            label = 'NEGATIVE'
        else:
            label = 'POSITIVE'
        if NEUTRAL_LOW <= confidence <= CONFIDENCE_THRESHOLD:
            label = 'NEUTRAL'
    else:
        label = list(model.config.id2label.values())[pred_idx]
    return label

print(f"\nNEUTRAL threshold: {NEUTRAL_LOW} - {CONFIDENCE_THRESHOLD}")
print(f"For 2-class model: confidence in [{NEUTRAL_LOW}, {CONFIDENCE_THRESHOLD}] = NEUTRAL")

Loading tokenizer...


Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Model label mapping: {0: '0', 1: '1'}
Number of labels: 2

NEUTRAL threshold: 0.4 - 0.6
For 2-class model: confidence in [0.4, 0.6] = NEUTRAL


In [4]:
from tqdm import tqdm

person_output_dir = os.path.join(OUTPUT_DIR, PERSON)
os.makedirs(person_output_dir, exist_ok=True)

def get_sentiment_batch(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True,
                       truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        scores = probs.max(dim=-1).values
    
    return preds.cpu().numpy(), scores.cpu().numpy()

CHECKPOINT_INTERVAL = 2000
predictions = []
confidences = []

print(f"Processing {len(comments_df)} comments...")
print(f"Batch size: {BATCH_SIZE}")

for i in tqdm(range(0, len(comments_df), BATCH_SIZE)):
    batch = comments_df['text_clean'].iloc[i:i+BATCH_SIZE].tolist()
    preds, scores = get_sentiment_batch(batch)
    predictions.extend(preds)
    confidences.extend(scores)
    
    if (i + BATCH_SIZE) % CHECKPOINT_INTERVAL == 0:
        checkpoint_df = comments_df.iloc[:i+BATCH_SIZE].copy()
        checkpoint_df['sentiment_idx'] = predictions[:i+BATCH_SIZE]
        checkpoint_df['sentiment_score'] = confidences[:i+BATCH_SIZE]
        checkpoint_df['sentiment_label'] = [convert_to_3class(p, c) for p, c in 
                                            zip(predictions[:i+BATCH_SIZE], confidences[:i+BATCH_SIZE])]
        checkpoint_path = os.path.join(person_output_dir, f'sentiment_checkpoint_{i+BATCH_SIZE}.csv')
        checkpoint_df.to_csv(checkpoint_path, index=False)
        print(f"\nCheckpoint saved: {checkpoint_path}")

comments_df['sentiment_idx'] = predictions
comments_df['sentiment_score'] = confidences
comments_df['sentiment_label'] = [convert_to_3class(p, c) for p, c in zip(predictions, confidences)]

print(f"\nSentiment distribution:")
print(comments_df['sentiment_label'].value_counts())
print(f"\nSample predictions:")
print(comments_df[['text_clean', 'sentiment_label', 'sentiment_score']].head(5))

Processing 16748 comments...
Batch size: 32


  0%|          | 0/524 [00:00<?, ?it/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Phase 4: Aggregation

### Aggregation Unit Decision Rule
```
if comment_count per slot > 1000:
    Use slot-level aggregation (21 windows)
else:
    Use daily aggregation (if timestamps available)
```

### Metrics Computed per Slot
| Metric | Description |
|--------|-------------|
| mean_sentiment | Average sentiment score (0-1) |
| pct_positive | % comments with POSITIVE label |
| pct_negative | % comments with NEGATIVE label |
| pct_neutral | % comments with NEUTRAL label |
| n_comments | Total comments in slot |

### Metrics Computed per Window
| Metric | Description |
|--------|-------------|
| mean_sentiment | Average across all slots in window |
| pct_positive | Overall % positive |
| pct_negative | Overall % negative |
| n_comments | Total comments in window |

In [5]:
# ============================================================
# Phase 4: Aggregation (Auto-detect structure)
# ============================================================

# Check if sentiment data exists
if 'sentiment_label' not in comments_df.columns or 'sentiment_score' not in comments_df.columns:
    print("ERROR: No sentiment data found. Run Phase 3 first.")
else:
    # Use auto-detected slot and window order
    slot_order = available_slots
    window_order = available_windows
    
    # Slot-level aggregation
    sentiment_by_slot = comments_df.groupby('slot').agg({
        'sentiment_score': 'mean'
    }).rename(columns={'sentiment_score': 'mean_sentiment'})
    
    sentiment_by_slot['n_comments'] = comments_df.groupby('slot')['comment_id'].count()
    
    for label in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
        sentiment_by_slot[f'pct_{label.lower()}'] = comments_df.groupby('slot')['sentiment_label'].apply(
            lambda x: (x == label).sum() / len(x) * 100
        )
    
    # Reorder using detected slots
    sentiment_by_slot = sentiment_by_slot.reindex([s for s in slot_order if s in sentiment_by_slot.index])
    
    print("\n=== Sentiment by Slot (chronological) ===")
    print(sentiment_by_slot.round(2))
    
    # Window-level aggregation
    sentiment_by_window = comments_df.groupby('window').agg({
        'sentiment_score': 'mean'
    }).rename(columns={'sentiment_score': 'mean_sentiment'})
    
    sentiment_by_window['n_comments'] = comments_df.groupby('window')['comment_id'].count()
    
    for label in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
        sentiment_by_window[f'pct_{label.lower()}'] = comments_df.groupby('window')['sentiment_label'].apply(
            lambda x: (x == label).sum() / len(x) * 100
        )
    
    # Reorder using detected windows
    sentiment_by_window = sentiment_by_window.reindex([w for w in window_order if w in sentiment_by_window.index])
    
    print("\n=== Sentiment by Window ===")
    print(sentiment_by_window.round(2))
    
    # Summary
    pre_mean = sentiment_by_window.loc['pre', 'mean_sentiment']
    cancel_mean = sentiment_by_window.loc['cancel', 'mean_sentiment']
    post_mean = sentiment_by_window.loc['post', 'mean_sentiment']
    
    print("\n=== Summary ===")
    print(f"Pre mean sentiment: {pre_mean:.4f}")
    print(f"Cancel mean sentiment: {cancel_mean:.4f}")
    print(f"Post mean sentiment: {post_mean:.4f}")
    print(f"Cancel vs Pre change: {((cancel_mean - pre_mean) / pre_mean * 100):.1f}%")
    print(f"Post vs Pre change: {((post_mean - pre_mean) / pre_mean * 100):.1f}%")
    
    # Export CSV files
    sentiment_by_slot.to_csv(os.path.join(person_output_dir, 'sentiment_by_slot.csv'))
    sentiment_by_window.to_csv(os.path.join(person_output_dir, 'sentiment_by_window.csv'))
    print(f"\nExported: {os.path.join(person_output_dir, 'sentiment_by_slot.csv')}")
    print(f"Exported: {os.path.join(person_output_dir, 'sentiment_by_window.csv')}")

## Phase 5: Recovery Analysis

### Recovery Definition
```
Recovery = sentiment >= (RECOVERY_THRESHOLD × pre_mean) for 3 consecutive post-cancel slots
```
- Default: RECOVERY_THRESHOLD = 0.9 (90% of pre_mean)
- Consecutive slots required: 3

### Key Metrics
| Metric | Description |
|--------|-------------|
| pre_mean | Mean sentiment during pre window (baseline) |
| cancel_mean | Mean sentiment during cancel window |
| nadir_slot | Slot with lowest sentiment |
| nadir_value | Minimum sentiment score |
| nadir_window | Which window contains the nadir (pre/cancel/post) |
| recovery_slot | First slot meeting recovery criteria |
| recovery_achieved | Boolean: recovery found or not |
| slots_to_recovery | Post slots until recovery |
| pct_change_cancel_vs_pre | % change from pre to cancel |
| pct_change_post_vs_pre | % change from pre to post |


In [6]:
# ============================================================
# Phase 5: Recovery Analysis
# ============================================================

# Configuration
RECOVERY_THRESHOLD = 0.9  # 90% of pre_mean
CONSECUTIVE_SLOTS = 3     # Required consecutive post-cancel slots

# Auto-detect pre/cancel/post slots from available_slots
pre_slots = [s for s in available_slots if 'pre' in s]
cancel_slots = [s for s in available_slots if 'cancel' in s]
post_slots = [s for s in available_slots if 'post' in s]

print(f"Detected slots - Pre: {len(pre_slots)}, Cancel: {len(cancel_slots)}, Post: {len(post_slots)}")

# Calculate baseline
pre_mean = sentiment_by_slot.loc[pre_slots, 'mean_sentiment'].mean() if pre_slots else None
cancel_mean = sentiment_by_slot.loc[cancel_slots, 'mean_sentiment'].mean() if cancel_slots else None
post_mean = sentiment_by_slot.loc[post_slots, 'mean_sentiment'].mean() if post_slots else None

print(f"Pre mean: {pre_mean:.4f}")
print(f"Cancel mean: {cancel_mean:.4f}")
print(f"Post mean: {post_mean:.4f}")

# Find nadir (minimum sentiment)
nadir_slot = sentiment_by_slot['mean_sentiment'].idxmin()
nadir_value = sentiment_by_slot['mean_sentiment'].min()

# Determine which window contains the nadir
if 'pre' in nadir_slot:
    nadir_window = 'pre'
elif 'cancel' in nadir_slot:
    nadir_window = 'cancel'
else:
    nadir_window = 'post'

print(f"Nadir: {nadir_slot} ({nadir_window}) = {nadir_value:.4f}")

# Find recovery (first slot where 3 consecutive post slots >= threshold)
recovery_threshold = pre_mean * RECOVERY_THRESHOLD
recovery_achieved = False
recovery_slot = None
slots_to_recovery = None

if post_slots:
    for i in range(len(post_slots) - CONSECUTIVE_SLOTS + 1):
        window = post_slots[i:i + CONSECUTIVE_SLOTS]
        window_values = [sentiment_by_slot.loc[s, 'mean_sentiment'] for s in window]
        if all(v >= recovery_threshold for v in window_values):
            recovery_achieved = True
            recovery_slot = window[0]
            slots_to_recovery = i + 1
            break

if recovery_achieved:
    print(f"Recovery achieved at: {recovery_slot} (slot {slots_to_recovery} of post-cancel)")
else:
    print("Recovery NOT achieved (3 consecutive post slots did not reach threshold)")

# Calculate percent changes
pct_change_cancel_vs_pre = ((cancel_mean - pre_mean) / pre_mean * 100) if pre_mean and cancel_mean else None
pct_change_post_vs_pre = ((post_mean - pre_mean) / pre_mean * 100) if pre_mean and post_mean else None

print(f"Change: Cancel vs Pre = {pct_change_cancel_vs_pre:.1f}%, Post vs Pre = {pct_change_post_vs_pre:.1f}%")

# Create recovery analysis DataFrame
recovery_data = {
    'metric': [
        'pre_mean', 'cancel_mean', 'post_mean',
        'nadir_slot', 'nadir_value', 'nadir_window',
        'recovery_slot', 'recovery_achieved', 'slots_to_recovery',
        'pct_change_cancel_vs_pre', 'pct_change_post_vs_pre',
        'recovery_threshold', 'consecutive_slots_required'
    ],
    'value': [
        pre_mean, cancel_mean, post_mean,
        nadir_slot, nadir_value, nadir_window,
        recovery_slot, recovery_achieved, slots_to_recovery,
        pct_change_cancel_vs_pre, pct_change_post_vs_pre,
        RECOVERY_THRESHOLD, CONSECUTIVE_SLOTS
    ]
}
recovery_df = pd.DataFrame(recovery_data).set_index('metric')

print("\n=== Recovery Analysis ===")
print(recovery_df)

# Export
recovery_df.to_csv(os.path.join(person_output_dir, 'recovery_analysis.csv'))
print(f"\nExported: {os.path.join(person_output_dir, 'recovery_analysis.csv')}")


## Phase 6: Visualization

### Figures Generated
| Figure | Description |
|--------|-------------|
| Sentiment trajectory | Line plot of mean_sentiment over slots. Nadir (red) and Recovery (green) markers |
| Sentiment by window | Bar chart: mean_sentiment pre/cancel/post |
| Sentiment distribution | Box plot: sentiment_score by window |

### Output Location
`03-analysis/figures/{PERSON}_sentiment_*.png`

In [7]:
# ============================================================
# Phase 6: Visualization
# ============================================================

# Check if sentiment data exists
if 'sentiment_label' not in comments_df.columns or 'sentiment_score' not in comments_df.columns:
    print("ERROR: No sentiment data found. Run Phase 3 first.")
else:
    os.makedirs(FIGURE_DIR, exist_ok=True)
    
    # Figure 1: Sentiment Trajectory (Line Plot)
    fig1, ax1 = plt.subplots(figsize=(14, 6))
    
    # Plot line
    ax1.plot(range(len(sentiment_by_slot)), sentiment_by_slot['mean_sentiment'], 
             marker='o', linewidth=2, markersize=6, color='steelblue', label='Mean Sentiment')
    
    # Color background by window
    for i, slot in enumerate(sentiment_by_slot.index):
        if 'pre' in slot:
            ax1.axvspan(i-0.5, i+0.5, alpha=0.1, color='blue')
        elif 'cancel' in slot:
            ax1.axvspan(i-0.5, i+0.5, alpha=0.2, color='red')
        else:
            ax1.axvspan(i-0.5, i+0.5, alpha=0.1, color='green')
    
    # Mark nadir
    nadir_idx = list(sentiment_by_slot.index).index(nadir_slot)
    ax1.scatter([nadir_idx], [nadir_value], color='red', s=200, zorder=5, 
                marker='v', label=f'Nadir: {nadir_slot}')
    
    # Mark recovery if achieved
    if recovery_achieved and recovery_slot:
        recovery_idx = list(sentiment_by_slot.index).index(recovery_slot)
        recovery_value = sentiment_by_slot.loc[recovery_slot, 'mean_sentiment']
        ax1.scatter([recovery_idx], [recovery_value], color='green', s=200, zorder=5,
                   marker='^', label=f'Recovery: {recovery_slot}')
    
    # Add threshold line
    ax1.axhline(y=pre_mean * RECOVERY_THRESHOLD, color='orange', linestyle='--', 
               alpha=0.7, label=f'Recovery Threshold ({RECOVERY_THRESHOLD*100}% of pre)')
    
    # Add window labels
    ax1.set_xticks(range(len(sentiment_by_slot)))
    ax1.set_xticklabels(sentiment_by_slot.index, rotation=45, ha='right')
    ax1.set_xlabel('Time Slot', fontsize=12)
    ax1.set_ylabel('Mean Sentiment Score', fontsize=12)
    ax1.set_title(f'{PERSON}: Sentiment Trajectory Over Time', fontsize=14, fontweight='bold')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, f'{PERSON}_sentiment_trajectory.png'), dpi=150)
    plt.close()
    print(f"Saved: {os.path.join(FIGURE_DIR, f'{PERSON}_sentiment_trajectory.png')}")
    
    # Figure 2: Sentiment by Window (Bar Chart)
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    
    window_colors = {'pre': 'steelblue', 'cancel': 'indianred', 'post': 'seagreen'}
    colors = [window_colors.get(w, 'gray') for w in sentiment_by_window.index]
    
    bars = ax2.bar(range(len(sentiment_by_window)), sentiment_by_window['mean_sentiment'], 
                  color=colors, edgecolor='black', linewidth=1.2)
    
    # Add value labels on bars
    for bar, val in zip(bars, sentiment_by_window['mean_sentiment']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11)
    
    # Add sample sizes
    for i, (idx, row) in enumerate(sentiment_by_window.iterrows()):
        ax2.text(i, 0.02, f'n={int(row["n_comments"])}', ha='center', va='bottom',
                fontsize=9, color='white', fontweight='bold')
    
    ax2.set_xticks(range(len(sentiment_by_window)))
    ax2.set_xticklabels([w.upper() for w in sentiment_by_window.index], fontsize=12)
    ax2.set_ylabel('Mean Sentiment Score', fontsize=12)
    ax2.set_title(f'{PERSON}: Sentiment by Window', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, max(sentiment_by_window['mean_sentiment']) * 1.15)
    ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, f'{PERSON}_sentiment_by_window.png'), dpi=150)
    plt.close()
    print(f"Saved: {os.path.join(FIGURE_DIR, f'{PERSON}_sentiment_by_window.png')}")
    
    # Figure 3: Sentiment Distribution (Box Plot)
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    
    # Prepare data for box plot
    window_order = [w for w in available_windows if w in comments_df['window'].unique()]
    box_data = [comments_df[comments_df['window'] == w]['sentiment_score'].dropna() for w in window_order]
    
    bp = ax3.boxplot(box_data, labels=[w.upper() for w in window_order], patch_artist=True)
    
    # Color the boxes
    for i, (patch, w) in enumerate(zip(bp['boxes'], window_order)):
        patch.set_facecolor(window_colors.get(w, 'gray'))
        patch.set_alpha(0.7)
    
    ax3.set_ylabel('Sentiment Score', fontsize=12)
    ax3.set_title(f'{PERSON}: Sentiment Distribution by Window', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_DIR, f'{PERSON}_sentiment_distribution.png'), dpi=150)
    plt.close()
    print(f"Saved: {os.path.join(FIGURE_DIR, f'{PERSON}_sentiment_distribution.png')}")
    
    print(f"\nAll visualizations saved to: {FIGURE_DIR}")

## Phase 7: Export

### Output Files
| File | Description | Status |
|--------|-------------|--------|
| sentiment_by_slot.csv | Metrics per slot | Already exported in Phase 4 |
| sentiment_by_window.csv | Metrics per window | Already exported in Phase 4 |
| recovery_analysis.csv | Recovery metrics | Already exported in Phase 5 |
| sentiment_comments.csv | All comments with sentiment scores | New export |

### Location
`03-analysis/outputs/{PERSON}/`

In [8]:
# ============================================================
# Phase 7: Export
# ============================================================

# Check if sentiment data exists
if 'sentiment_label' not in comments_df.columns or 'sentiment_score' not in comments_df.columns:
    print("ERROR: No sentiment data found. Run Phase 3 first.")
else:
    os.makedirs(person_output_dir, exist_ok=True)
    
    exported_files = []
    missing_files = []
    
    # Export sentiment_comments.csv (the main new export)
    comments_export_path = os.path.join(person_output_dir, 'sentiment_comments.csv')
    if os.path.exists(comments_export_path):
        print(f"Already exists: sentiment_comments.csv")
    else:
        # Ensure all required columns exist
        export_cols = ['comment_id', 'parent_id', 'author', 'author_id', 'timestamp', 
                      'like_count', 'text', 'video_id', 'window', 'slot', 'text_clean',
                      'sentiment_idx', 'sentiment_score', 'sentiment_label']
        # Only export columns that exist
        available_cols = [c for c in export_cols if c in comments_df.columns]
        comments_df[available_cols].to_csv(comments_export_path, index=False)
        exported_files.append('sentiment_comments.csv')
        print(f"Exported: {comments_export_path}")
    
    # Check what else might be missing
    check_files = ['sentiment_by_slot.csv', 'sentiment_by_window.csv', 'recovery_analysis.csv']
    for f in check_files:
        fpath = os.path.join(person_output_dir, f)
        if os.path.exists(fpath):
            print(f"Already exists: {f}")
        else:
            print(f"WARNING: Missing {f} - run earlier phases first")
    
    # Summary
    print("\n" + "="*50)
    print("EXPORT SUMMARY")
    print("="*50)
    print(f"Person: {PERSON}")
    print(f"Output directory: {person_output_dir}")
    print(f"Total comments: {len(comments_df)}")
    print(f"\nFiles exported in this run: {len(exported_files)}")
    for f in exported_files:
        print(f"  - {f}")

## Summary

### RQ1 Answer Framework
Sentiment trajectory analysis provides:
- Baseline sentiment (pre-mean)
- Impact of cancel event (nadir value, nadir timing)
- Recovery pattern (recovery time, persistence)

### Outputs Generated
- Trajectory visualization showing pre/cancel/post patterns
- Quantitative metrics for statistical comparison
- Recovery analysis to answer how audiences recover